In [2]:
import ollama
import pandas as pd
from yahoo_finance import MarketData
from news_embedder import NewsEmbedder

OLLAMA_HOST = "http://100.68.136.71:11434"

# Verifica se o modelo já está disponível antes de puxar
client = ollama.Client(host=OLLAMA_HOST)


In [3]:
client.list().models

11:37:08 [INFO] HTTP Request: GET http://100.68.136.71:11434/api/tags "HTTP/1.1 200 OK"


[Model(model='qwen3-embedding:4b', modified_at=datetime.datetime(2026, 3, 16, 11, 3, 45, 563525, tzinfo=TzInfo(-10800)), digest='df5bd2e3c74cd8d069d21dc038f1b359fcdc9458fce1c99bd43c9eb1518ff907', size=2496704041, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='4.0B', quantization_level='Q4_K_M')),
 Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 16, 31, 728797, tzinfo=TzInfo(-10800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')),
 Model(model='lfm2.5-thinking:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 12, 22, 506453, tzinfo=TzInfo(-10800)), digest='95bd9d45385f33bfe96d8b3651c8569e152f21f5bdb7c19894ffde650e9cf140', size=731163903, details=ModelDetails(parent_model='', format='gguf', family='lfm2', familie

In [ ]:
embedding_model = "qwen3-embedding:4b"

In [4]:
modelos = [m.model for m in client.list().models]
if embedding_model not in modelos:
    print(f"Baixando {embedding_model}...")
    client.pull(embedding_model)

In [ ]:


# Notícias
df_news = pd.read_json("../1.news/itub4_noticias.json", convert_dates=False)
articles  = df_news.to_dict(orient="records")

# Preços
md      = MarketData("ITUB4.SA")
X_price = md.features(period="1y", lags=5)
y       = md.target(period="1y", horizon=5)

# Embedder
ne = NewsEmbedder(
    model=embedding_model,
    fields=["title", "excerpt", "content"],
    ollama_host=OLLAMA_HOST,
    cache_path="embeddings_cache.npz",
)



In [1]:
# Dataset final
X_full = ne.merge_with_prices(X_price, articles)
df     = X_full.join(y).dropna()
X      = df.drop(columns=["target"])
y      = df["target"]

print(f"Shape final: {X.shape}")  # (dias, price_features + 768 emb dims)

NameError: name 'ne' is not defined

In [10]:
X_full.to_csv("dataset_full.csv", index=False)